In [1]:
import random
from google.colab import files

# --- Настройки ---
DB_NAME_PLACEHOLDER = "mgpu_ico_db_09"
NUM_COURSES = random.randint(120, 150)  # 120-150 курсов
NUM_STUDENTS = random.randint(120, 150)  # 120-150 студентов
OUTPUT_SQL_FILENAME = "populate_courses_students_large.sql"

# --- Расширенные данные для генерации ---

# Расширенный словарь предметов и курсов
subjects_courses = {
    "Химия": ["Интенсивный курс химии", "Химия для начинающих", "Органическая химия", "Биохимия", "Аналитическая химия", "Физическая химия", "Химия полимеров", "Неорганическая химия"],
    "Аналитика": ["Анализ данных с нуля", "Занимательная аналитика", "Современные методы анализа данных", "Бизнес-аналитика", "Визуализация данных", "Статистический анализ", "Аналитика в Excel", "Big Data аналитика"],
    "Математика": ["Теория вероятностей", "Математический анализ", "Линейная алгебра", "Дискретная математика", "Дифференциальные уравнения", "Теория чисел", "Математическая логика", "Вычислительная математика"],
    "Философия": ["Занимательная философия", "История философии", "Философия науки", "Этическая философия", "Политическая философия", "Восточная философия", "Современная философия"],
    "Английский язык": ["Мастер-класс по английскому языку", "Профессиональный английский язык", "Деловой английский", "Английский для IT", "Разговорный английский", "Английская грамматика", "Подготовка к IELTS"],
    "Программирование": ["Программирование для новичков", "Интенсивный курс программирования", "Python для начинающих", "Java программирование", "Web-разработка", "Мобильная разработка", "Базы данных и SQL", "Алгоритмы и структуры данных"],
    "Информатика": ["Основы кибербезопасности", "Информатика в повседневной жизни", "Сетевые технологии", "Операционные системы", "Искусственный интеллект", "Машинное обучение", "Компьютерные сети", "Криптография"],
    "История": ["В мире истории", "Профессиональная история", "История Древнего мира", "Средневековая история", "История России", "Всемирная история", "История искусств", "Археология"],
    "Маркетинг": ["Профессиональный маркетинг", "Цифровой маркетинг", "SMM маркетинг", "Контент-маркетинг", "Email-маркетинг", "Маркетинговые исследования", "Бренд-менеджмент"],
    "Экономика": ["Такая разная экономика", "Мастер-классы для бухгалтеров", "Основы финансовой грамотности", "Микроэкономика", "Макроэкономика", "Финансовый менеджмент", "Банковское дело", "Международная экономика"],
    "Психология": ["Курсы для начинающих психологов", "Профессиональная психология", "Клиническая психология", "Социальная психология", "Когнитивная психология", "Детская психология", "Психология личности"]
}

# Расширенные списки имен и фамилий
male_first_names = ["Иван", "Алексей", "Дмитрий", "Сергей", "Михаил", "Павел", "Андрей", "Владимир", "Глеб", "Петр", "Александр", "Максим", "Артем", "Кирилл", "Егор", "Роман", "Никита", "Даниил", "Тимофей", "Игорь"]
female_first_names = ["Мария", "Елена", "Ольга", "Анна", "Екатерина", "Наталья", "Татьяна", "Ксения", "Юлия", "Светлана", "Анастасия", "Дарья", "Алина", "Ирина", "Виктория", "Полина", "София", "Александра", "Валерия", "Маргарита"]
last_names = ["Иванов", "Петров", "Сидоров", "Кузнецов", "Смирнов", "Васильев", "Морозов", "Лебедев", "Козлов", "Новиков", "Федоров", "Соколов", "Попов", "Волков", "Зайцев", "Павлов", "Семенов", "Голубев", "Виноградов", "Богданов"]

# Дополнительные домены для email
email_domains = ["gmail.com", "yandex.ru", "mail.ru", "outlook.com", "icloud.com", "hotmail.com", "protonmail.com", "rambler.ru"]

# --- Функции для генерации SQL-запросов ---

def generate_courses_insert_statements():
    """Генерирует INSERT запросы для таблицы courses."""
    inserts = []
    course_id = 1

    # Создаем расширенный список всех возможных курсов
    all_courses = []
    for subject, courses_list in subjects_courses.items():
        for course_title in courses_list:
            all_courses.append((subject, course_title))

    # Генерируем 120-150 уникальных курсов
    for i in range(NUM_COURSES):
        subject, base_course_title = random.choice(all_courses)

        # Создаем вариации названий курсов
        variations = ["", " (базовый)", " (продвинутый)", " (интенсив)", " (профессиональный)", " (экспресс)", " (полный курс)"]
        variation = random.choice(variations)
        course_title = f"{base_course_title}{variation}"

        # Генерация цены от 5000 до 50000 рублей
        price = round(random.uniform(5000, 50000), 2)

        inserts.append(
            f"INSERT INTO courses (course_id, title, subject, price) VALUES "
            f"({course_id}, '{course_title}', '{subject}', {price});"
        )
        course_id += 1

    return "\n".join(inserts)

def generate_students_insert_statements(num_records):
    """Генерирует INSERT запросы для таблицы students."""
    inserts = []

    used_emails = set()  # Для отслеживания уникальных email

    for student_id in range(1, num_records + 1):
        # Генерация уникального email
        attempts = 0
        while attempts < 10:  # Защита от бесконечного цикла
            # Выбор пола и имени
            if random.random() < 0.5:  # 50% шанс мужского имени
                first_name = random.choice(male_first_names)
                last_name = random.choice(last_names)
                gender_suffix = ""
            else:  # 50% шанс женского имени
                first_name = random.choice(female_first_names)
                last_name = random.choice(last_names) + "а"  # Женская форма фамилии
                gender_suffix = "a"

            # Генерация email с номером для уникальности
            email_username = f"{first_name.lower()}.{last_name.lower()}"
            email_domain = random.choice(email_domains)

            # Добавляем случайный номер если email уже используется
            if f"{email_username}@{email_domain}" in used_emails:
                email_username = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 999)}"

            email = f"{email_username}@{email_domain}"

            if email not in used_emails:
                used_emails.add(email)
                break
            attempts += 1

        inserts.append(
            f"INSERT INTO students (student_id, first_name, email) VALUES "
            f"({student_id}, '{first_name} {last_name}', '{email}');"
        )

    return "\n".join(inserts)

# --- Основная логика скрипта ---
sql_script_content = []
sql_script_content.append(f"-- SQL-скрипт для заполнения БД '{DB_NAME_PLACEHOLDER}'\n")
sql_script_content.append(f"USE {DB_NAME_PLACEHOLDER};\n")

sql_script_content.append("-- Очистка таблиц перед вставкой новых данных\n")
sql_script_content.append("SET FOREIGN_KEY_CHECKS = 0;")
sql_script_content.append("TRUNCATE TABLE courses;")
sql_script_content.append("TRUNCATE TABLE students;")
sql_script_content.append("SET FOREIGN_KEY_CHECKS = 1;\n")

sql_script_content.append("-- Заполнение таблицы 'courses'\n")
sql_script_content.append(generate_courses_insert_statements())
sql_script_content.append("\n-- Заполнение таблицы 'students'\n")
sql_script_content.append(generate_students_insert_statements(NUM_STUDENTS))

final_sql_string = "\n".join(sql_script_content)

# Создание и скачивание файла
with open(OUTPUT_SQL_FILENAME, "w", encoding="utf-8") as f:
    f.write(final_sql_string)

print(f"SQL-скрипт '{OUTPUT_SQL_FILENAME}' успешно сгенерирован.")
print(f"Создано {NUM_COURSES} курсов и {NUM_STUDENTS} студентов.")
print("Сейчас начнется скачивание файла...")

files.download(OUTPUT_SQL_FILENAME)

SQL-скрипт 'populate_courses_students_large.sql' успешно сгенерирован.
Создано 133 курсов и 125 студентов.
Сейчас начнется скачивание файла...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>